In [3]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.multioutput import MultiOutputClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.model_selection import GridSearchCV
import re
from nltk.stem import WordNetLemmatizer
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import uniform, randint

#Carga de datos
train_data = pd.read_csv("../../Data/train_indexado.csv")
test_data = pd.read_csv("../../Data/test_indexado.csv")

# Definir las clases de emociones
emotion_classes = train_data.columns[2:].tolist()

# --- PREPROCESAMIENTO PARA LEMATIZACIÓN ---
lemmatizer = WordNetLemmatizer()

def preprocess_text(text):
    words = re.findall(r'\b\w+\b', text.lower())
    lemmatized_words = [lemmatizer.lemmatize(word) for word in words]
    return " ".join(lemmatized_words)

X_train_lem = train_data['Text'].apply(preprocess_text)
X_test_lem = test_data['Text'].apply(preprocess_text)

# TF-IDF VECTORIZACIÓN
vectorizer = TfidfVectorizer(lowercase=True, strip_accents="unicode", max_features=10000)
X_train = vectorizer.fit_transform(X_train_lem)
X_test = vectorizer.transform(X_test_lem)
y_train = np.asarray(train_data[emotion_classes])
y_test = np.asarray(test_data[emotion_classes])

# Selección de features con Chi2 = 500
selector = SelectKBest(score_func=chi2, k=500)
X_train_chi = selector.fit_transform(X_train, y_train)
X_test_chi = selector.transform(X_test)

print(f"Datos preparados: {X_train_chi.shape[0]} muestras de entrenamiento, {X_train_chi.shape[1]} features")

Datos preparados: 43410 muestras de entrenamiento, 500 features


In [ ]:
# Hiperparametrización de Random Forest con Chi2=500
print("Iniciando búsqueda de hiperparámetros para Random Forest (Chi2=500)...")

# Parámetros a probar
param_distributions = {
    'estimator__n_estimators': randint(150, 350),    
    'estimator__max_depth': [15, 25, None],          
    'estimator__min_samples_split': randint(2, 6),
    'estimator__min_samples_leaf': randint(1, 3),
    'estimator__max_features': ['sqrt', 'log2']
}

# Crear el modelo base
rf_base = RandomForestClassifier(random_state=42)
multi_rf = MultiOutputClassifier(rf_base)

# GridSearchCV
random_search = RandomizedSearchCV(
    multi_rf,
    param_distributions=param_distributions,
    n_iter=20,  # puedes ajustar
    scoring="f1_macro",
    cv=3,
    verbose=2,
    random_state=42,
    n_jobs=-1
)

print(f"Total de combinaciones a probar: {random_search.n_iter}")
# Entrenar
random_search.fit(X_train_chi, y_train)



Iniciando búsqueda de hiperparámetros para Random Forest (Chi2=500)...
Total de combinaciones a probar: 20
Fitting 3 folds for each of 20 candidates, totalling 60 fits
[CV] END estimator__max_depth=15, estimator__max_features=sqrt, estimator__min_samples_leaf=1, estimator__min_samples_split=3, estimator__n_estimators=224; total time= 6.0min
[CV] END estimator__max_depth=15, estimator__max_features=sqrt, estimator__min_samples_leaf=1, estimator__min_samples_split=3, estimator__n_estimators=224; total time= 6.3min
[CV] END estimator__max_depth=15, estimator__max_features=sqrt, estimator__min_samples_leaf=1, estimator__min_samples_split=3, estimator__n_estimators=224; total time= 6.3min
[CV] END estimator__max_depth=None, estimator__max_features=log2, estimator__min_samples_leaf=1, estimator__min_samples_split=5, estimator__n_estimators=253; total time=20.5min
[CV] END estimator__max_depth=None, estimator__max_features=log2, estimator__min_samples_leaf=1, estimator__min_samples_split=5, e

/opt/anaconda3/lib/python3.12/site-packages/joblib/externals/loky/process_executor.py:752: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


[CV] END estimator__max_depth=None, estimator__max_features=log2, estimator__min_samples_leaf=1, estimator__min_samples_split=3, estimator__n_estimators=237; total time=19.5min
[CV] END estimator__max_depth=15, estimator__max_features=sqrt, estimator__min_samples_leaf=2, estimator__min_samples_split=3, estimator__n_estimators=171; total time= 4.3min
[CV] END estimator__max_depth=None, estimator__max_features=log2, estimator__min_samples_leaf=1, estimator__min_samples_split=5, estimator__n_estimators=253; total time=20.4min
[CV] END estimator__max_depth=25, estimator__max_features=log2, estimator__min_samples_leaf=2, estimator__min_samples_split=5, estimator__n_estimators=337; total time= 6.9min
[CV] END estimator__max_depth=25, estimator__max_features=log2, estimator__min_samples_leaf=2, estimator__min_samples_split=5, estimator__n_estimators=337; total time= 7.3min
[CV] END estimator__max_depth=25, estimator__max_features=log2, estimator__min_samples_leaf=2, estimator__min_samples_spl

NameError: name 'grid_search' is not defined

In [6]:
print(f"\nMejores parámetros: {random_search.best_params_}")
print(f"Mejor score CV (F1 macro): {random_search.best_score_:.5f}")


Mejores parámetros: {'estimator__max_depth': None, 'estimator__max_features': 'sqrt', 'estimator__min_samples_leaf': 2, 'estimator__min_samples_split': 2, 'estimator__n_estimators': 313}
Mejor score CV (F1 macro): 0.28700


In [7]:

# Evaluar el mejor modelo
best_rf = random_search.best_estimator_
y_pred_best = best_rf.predict(X_test_chi)

# Métricas del mejor modelo
report_dict_best = classification_report(y_test, y_pred_best, output_dict=True, zero_division=0)
f1_macro_best = report_dict_best["macro avg"]["f1-score"]
recall_macro_best = report_dict_best["macro avg"]["recall"]

print("\n=== RESULTADOS DEL MEJOR RANDOM FOREST (Chi2=500) ===")
print(f"Accuracy: {accuracy_score(y_test, y_pred_best):.5f}")
print(f"F1 Score (macro avg): {f1_macro_best:.5f}")
print(f"Recall Score (macro avg): {recall_macro_best:.5f}")
print(f"\nClassification Report:\n{classification_report(y_test, y_pred_best, zero_division=0)}")

# Comparar con Random Forest por defecto (Chi2=500)
rf_default = RandomForestClassifier(random_state=42)
multi_rf_default = MultiOutputClassifier(rf_default)
multi_rf_default.fit(X_train_chi, y_train)
y_pred_default = multi_rf_default.predict(X_test_chi)

report_dict_default = classification_report(y_test, y_pred_default, output_dict=True, zero_division=0)
f1_macro_default = report_dict_default["macro avg"]["f1-score"]
recall_macro_default = report_dict_default["macro avg"]["recall"]

print("\n=== COMPARACIÓN CON RANDOM FOREST POR DEFECTO (Chi2=500) ===")
print(f"Accuracy por defecto: {accuracy_score(y_test, y_pred_default):.5f}")
print(f"F1 Score por defecto (macro avg): {f1_macro_default:.5f}")
print(f"Recall Score por defecto (macro avg): {recall_macro_default:.5f}")

print("\n=== MEJORA OBTENIDA ===")
print(f"Mejora en Accuracy: {accuracy_score(y_test, y_pred_best) - accuracy_score(y_test, y_pred_default):.5f}")
print(f"Mejora en F1 Score: {f1_macro_best - f1_macro_default:.5f}")
print(f"Mejora en Recall: {recall_macro_best - recall_macro_default:.5f}")

# Mostrar configuración óptima
print("\n=== CONFIGURACIÓN ÓPTIMA ===")
print(f"Chi2 features: 500")
for param, value in random_search.best_params_.items():
    print(f"{param}: {value}")


=== RESULTADOS DEL MEJOR RANDOM FOREST (Chi2=500) ===
Accuracy: 0.31933
F1 Score (macro avg): 0.30206
Recall Score (macro avg): 0.23648

Classification Report:
              precision    recall  f1-score   support

           0       0.71      0.45      0.55       504
           1       0.76      0.75      0.76       264
           2       0.62      0.12      0.20       198
           3       0.77      0.05      0.10       320
           4       0.73      0.09      0.17       351
           5       0.73      0.06      0.11       135
           6       0.71      0.08      0.14       153
           7       0.90      0.03      0.06       284
           8       0.83      0.12      0.21        83
           9       0.75      0.06      0.11       151
          10       0.38      0.01      0.02       267
          11       0.76      0.20      0.32       123
          12       0.57      0.11      0.18        37
          13       0.79      0.22      0.35       103
          14       0.82     